# Pipeline Debug Walkthrough
Step through one sample: **load → technicals → prompt → decision → outcome**.

Set `SYMBOL`, `DATE`, `SLOT` in the first cell then **Run All**.


In [ ]:
# ── Parameters ─────────────────────────────────────────────────
SYMBOL     = "CANBK"        # NSE ticker
DATE       = "2024-08-19"   # decision date  YYYY-MM-DD
SLOT       = "open"         # "open" | "mid" | "close"
N_1H       = 90             # 1H candles to load (≈18 sessions)
N_5M       = 90             # 5min candles to load (≈7.5 hours)
CHART_BARS = 40             # bars shown in per-TF annotated charts


In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath("."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import matplotlib.transforms as mtransforms

# ── TradingView-inspired dark palette ───────────────────────────
C = dict(
    bg="#131722", grid="#363a45", text="#d1d4dc",
    up="#26a69a",  dn="#ef5350",
    e9="#f6c90e", e21="#ff9800", e50="#f44336",
    sup="#26a69a", res="#ef5350",
    vwap="#b39ddb", entry="#42a5f5",
    sl="#ef5350",  tp="#26a69a",
    sh="#f6c90e",  sw="#7e57c2",
)
plt.rcParams.update({
    "figure.facecolor": C["bg"],  "axes.facecolor":  C["bg"],
    "axes.edgecolor":   C["grid"],"text.color":       C["text"],
    "xtick.color":      C["text"],"ytick.color":      C["text"],
    "grid.color":       C["grid"],"grid.linewidth":   0.5,
    "font.size": 9, "figure.dpi": 130,
})

# ── Chart helpers ───────────────────────────────────────────────
def _candles(ax, df):
    for i, (_, r) in enumerate(df.iterrows()):
        col = C["up"] if r.close >= r.open else C["dn"]
        lo, hi = min(r.open, r.close), max(r.open, r.close)
        ax.add_patch(Rectangle((i - 0.4, lo), 0.8, max(hi - lo, 0.001),
                                color=col, zorder=3))
        ax.plot([i, i], [r.low, r.high], color=col, lw=0.7, zorder=2)
    ax.set_xlim(-0.8, len(df) + 0.2)
    ax.autoscale_view()
    n = len(df); step = max(1, n // 8); ticks = list(range(0, n, step))
    ax.set_xticks(ticks)
    ax.set_xticklabels([df.index[i].strftime("%m-%d %H:%M") for i in ticks],
                        rotation=30, ha="right", fontsize=7)
    ax.yaxis.tick_right(); ax.yaxis.set_label_position("right")
    ax.grid(alpha=0.35)

def _hline(ax, y, color, label="", lw=1.0, ls="--"):
    ax.axhline(y, color=color, lw=lw, ls=ls, alpha=0.85, zorder=4)
    if label:
        trans = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
        ax.text(0.004, y, f"  {label}", transform=trans,
                color=color, fontsize=7.5, va="bottom", zorder=5)

def _xi(df, date_str):
    target = pd.Timestamp(date_str)
    diffs  = [(abs((ts - target).total_seconds()), i) for i, ts in enumerate(df.index)]
    return min(diffs)[1]

def _sw_marks(ax, points, chart_df, marker, color, dy):
    chart_start = chart_df.index[0]
    for s in (points or []):
        ts = pd.Timestamp(s["date"])
        if ts >= chart_start and ts <= chart_df.index[-1]:
            xi = _xi(chart_df, s["date"])
            ax.plot(xi, s["price"], marker, color=color, ms=7, zorder=6)
            ax.text(xi, s["price"] + dy, f"{s['price']:.1f}",
                    color=color, fontsize=6.5, ha="center", va="bottom", zorder=6)

from technicals import TechnicalsAnalyzer
from prompts import build_prompt_mtf, SYSTEM_PROMPT
ta = TechnicalsAnalyzer()
print("Imports OK — ta ready")


In [ ]:
# ══════════════════════════════════════════════════════════════
print(f"STEP 1 · Load intraday candle history")
print(f"  {SYMBOL}  {DATE}  @{SLOT}  |  1H={N_1H} bars  5M={N_5M} bars")
print("─" * 56)

df_1h = ta.load_intraday_for_slot(SYMBOL, DATE, "1h",   N_1H, SLOT)
df_5m = ta.load_intraday_for_slot(SYMBOL, DATE, "5min", N_5M, SLOT)
print(f"  1H : {len(df_1h)} bars   {df_1h.index[0] if not df_1h.empty else '—'}  →  {df_1h.index[-1] if not df_1h.empty else '—'}")
print(f"  5M : {len(df_5m)} bars   {df_5m.index[0] if not df_5m.empty else '—'}  →  {df_5m.index[-1] if not df_5m.empty else '—'}")
if not df_1h.empty:
    print(f"  1H last close : {df_1h['close'].iloc[-1]:.2f}")
if not df_5m.empty:
    print(f"  5M last close : {df_5m['close'].iloc[-1]:.2f}")

# ── Side-by-side 1H and 5M OHLCV charts ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"Step 1 · {SYMBOL} · Intraday candles  [{DATE} @{SLOT}]",
             color=C["text"], fontsize=11)
for ax, df_tf, label in [(axes[0], df_1h, f"1H ({N_1H} bars)"),
                          (axes[1], df_5m, f"5M ({N_5M} bars)")]:
    ax.set_title(label, color=C["text"], fontsize=10)
    if not df_tf.empty:
        _candles(ax, df_tf.tail(min(CHART_BARS, len(df_tf))))
    else:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                color=C["text"], ha="center", va="center")
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
print("STEP 2 · Compute technical indicators (1H + 5M)")
print("─" * 56)

tf_1h = ta.compute_tf(df_1h, spans=(9, 21, 50), label="1h", pivot_order=5)
tf_5m = ta.compute_tf(df_5m, spans=(9, 21, 50), label="5m", pivot_order=2)
vc    = ta.vix_context(DATE)
zc    = ta.compute_zones(SYMBOL, DATE)

print("  ── 1H (long bias) ──────────────────────────")
print(f"  Close  : {tf_1h['close']:.2f}    VWAP(sess): {tf_1h['vwap']}")
print(f"  EMA9:{tf_1h['ema_fast']}  EMA21:{tf_1h['ema_mid']}  EMA50:{tf_1h['ema_slow']}")
print(f"  Stack  : {tf_1h['ema_stack']}")
print(f"  Trend  : {tf_1h['trend']}   Struct: {tf_1h.get('structure') or ['-']}")
ns1 = tf_1h.get("nearest_support"); nr1 = tf_1h.get("nearest_resistance")
print(f"  Nearest S: {ns1['mid'] if ns1 else '—'}   Nearest R: {nr1['mid'] if nr1 else '—'}")
print(f"  PDH/PDL : {tf_1h.get('levels', {})}")
print()
print("  ── 5M (short bias) ─────────────────────────")
print(f"  Close  : {tf_5m['close']:.2f}    VWAP(sess): {tf_5m['vwap']}")
print(f"  EMA9:{tf_5m['ema_fast']}  EMA21:{tf_5m['ema_mid']}  EMA50:{tf_5m['ema_slow']}")
print(f"  Stack  : {tf_5m['ema_stack']}")
print(f"  Trend  : {tf_5m['trend']}   Struct: {tf_5m.get('structure') or ['-']}")
ns5 = tf_5m.get("nearest_support"); nr5 = tf_5m.get("nearest_resistance")
print(f"  Nearest S: {ns5['mid'] if ns5 else '—'}   Nearest R: {nr5['mid'] if nr5 else '—'}")
print()
if vc: print(f"  VIX: {vc['vix']}  ({vc['regime']})  pctile_1y={vc['pctile_1y']}%")

# Look-ahead check
rc1 = tf_1h.get("recent_candles") or []
ok1 = (rc1[-1]['date'] < DATE) if rc1 and SLOT in ("open","mid") else True
rc5 = tf_5m.get("recent_candles") or []
ok5 = (str(rc5[-1]['date'])[:10] < DATE) if rc5 and SLOT in ("open","mid") else True
print(f"\n  Look-ahead 1H: {'OK' if ok1 else '⚠ FAIL'}  (last={rc1[-1]['date'] if rc1 else '—'})")
print(f"  Look-ahead 5M: {'OK' if ok5 else '⚠ FAIL'}  (last={rc5[-1]['date'] if rc5 else '—'})")


In [ ]:
def _tf_chart(ax, df_tf, tf, label):
    """Draw annotated technical chart for one timeframe."""
    chart = df_tf.tail(min(CHART_BARS, len(df_tf)))
    xs    = list(range(len(chart)))

    # Candles
    _candles(ax, chart)
    ax.set_title(label, color=C["text"], fontsize=10)

    # EMAs (recomputed on full df for smooth lines, sliced to chart window)
    sp = tf.get("spans", [9, 21, 50])
    for span, col, key in zip(sp, [C["e9"], C["e21"], C["e50"]],
                               ["ema_fast", "ema_mid", "ema_slow"]):
        ema = df_tf["close"].ewm(span=span, adjust=False).mean().loc[chart.index]
        ax.plot(xs, ema.values, color=col, lw=1.3, label=f"EMA{span} {tf[key]:.1f}")

    # Session VWAP
    if tf.get("vwap"):
        ax.axhline(tf["vwap"], color=C["vwap"], lw=1.1, ls="-.", alpha=0.85,
                   label=f"VWAP {tf['vwap']:.2f}")

    # Support / resistance zones (shaded)
    for z in (tf.get("support_zones") or []):
        if z:
            lo = z["low"]; hi = max(z["high"], z["low"] + 0.1)
            ax.axhspan(lo - 0.01, hi + 0.01, color=C["sup"], alpha=0.09, zorder=1)
    for z in (tf.get("resistance_zones") or []):
        if z:
            lo = z["low"]; hi = max(z["high"], z["low"] + 0.1)
            ax.axhspan(lo - 0.01, hi + 0.01, color=C["res"], alpha=0.09, zorder=1)

    # Nearest S/R lines
    ns = tf.get("nearest_support"); nr = tf.get("nearest_resistance")
    if ns: _hline(ax, ns["mid"], C["sup"], f"S {ns['mid']:.2f}", lw=1.4)
    if nr: _hline(ax, nr["mid"], C["res"], f"R {nr['mid']:.2f}", lw=1.4)

    # PDH / PDL (1H only)
    lv = tf.get("levels") or {}
    if lv.get("pdh"): _hline(ax, lv["pdh"], "#90a4ae", f"PDH {lv['pdh']}", lw=0.8, ls=":")
    if lv.get("pdl"): _hline(ax, lv["pdl"], "#90a4ae", f"PDL {lv['pdl']}", lw=0.8, ls=":")

    # Swing marks (only those within the chart window)
    dy_up = (chart["high"].max() - chart["low"].min()) * 0.012
    _sw_marks(ax, tf.get("swing_highs_30d") or tf.get("swing_highs"),
              chart, "^", C["sh"],  dy= dy_up)
    _sw_marks(ax, tf.get("swing_lows_30d")  or tf.get("swing_lows"),
              chart, "v", C["sw"],  dy=-dy_up * 2)

    # Structure label
    struct = tf.get("structure") or []
    if struct:
        ax.text(0.98, 0.02, "  ".join(struct), transform=ax.transAxes,
                color=C["text"], fontsize=8.5, ha="right", va="bottom", alpha=0.7)

    # Bias chip
    stack  = tf.get("ema_stack", "")
    bcol   = C["up"] if stack.startswith("bullish") else C["dn"] if stack.startswith("bearish") else C["text"]
    bias   = "BULL" if stack.startswith("bullish") else "BEAR" if stack.startswith("bearish") else "MIX"
    ax.text(0.01, 0.97, f"{bias}", transform=ax.transAxes, color=bcol,
            fontsize=11, fontweight="bold", va="top", alpha=0.85)

    ax.legend(loc="lower left", fontsize=7.5, framealpha=0.3,
              facecolor=C["bg"], edgecolor=C["grid"])


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 7))
fig.suptitle(f"Step 2 · {SYMBOL} · Technical setup · {DATE} @{SLOT}",
             color=C["text"], fontsize=11)
_tf_chart(ax1, df_1h, tf_1h, f"1H ({N_1H} bars)  LONG")
_tf_chart(ax2, df_5m, tf_5m, f"5M ({N_5M} bars)  SHORT")

# VIX tag on 1H chart
if vc:
    ax1.text(0.01, 0.02, f"VIX {vc['vix']} · {vc['regime']} · {vc.get('dir_5d','')}",
             transform=ax1.transAxes, color=C["vwap"], fontsize=8, va="bottom", alpha=0.8)

plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
print("STEP 3 · Prompt fed to the LLM")
print("─" * 56)
prompt_text = build_prompt_mtf(SYMBOL, DATE, tf_1h, tf_5m, vc or {}, zc or {}, SLOT)
print(prompt_text)


In [ ]:
# ══════════════════════════════════════════════════════════════
print("STEP 4 · LLM / rule-based decision")
print("─" * 56)
cur_price = tf_5m.get("close") or tf_1h.get("close", 0)
try:
    from pipeline import decide
    dec, _ = decide(SYMBOL, DATE, tf_1h, tf_5m, vc or {}, zc or {}, decision_slot=SLOT)
except Exception as _err:
    print(f"[LLM unavailable: {_err}]  — storing HOLD placeholder")
    dec = {"action": "HOLD", "planned_entry": cur_price,
           "stop_loss": 0, "take_profit": 0, "risk_reward": 0,
           "confidence": 0, "reasoning": "LLM not reachable", "features_used": []}

action = dec.get("action", "HOLD")
entry  = float(dec.get("planned_entry") or cur_price)
sl     = float(dec.get("stop_loss")     or 0)
tp     = float(dec.get("take_profit")   or 0)
conf   = float(dec.get("confidence")    or 0)
rr     = float(dec.get("risk_reward")   or 0)

print(json.dumps(dec, indent=2))
print()
print(f"  ACTION : {action}")
print(f"  Entry  : {entry:.2f}  (1H close={tf_1h['close']}  5M close={tf_5m['close']})")
if sl: print(f"  SL     : {sl:.2f}  ({(sl - entry) / entry * 100:+.2f}%)")
if tp: print(f"  TP     : {tp:.2f}  ({(tp - entry) / entry * 100:+.2f}%)")
print(f"  Conf   : {conf:.2f}    R:R : {rr:.2f}")


In [ ]:
# ── Entry / SL / TP chart (last 20 5M bars — finest resolution) ─
dec_df = df_5m.tail(20)

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle(
    f"Step 5 · {SYMBOL} (5M) · {action}  "
    + (f"entry={entry:.2f}  SL={sl:.2f}  TP={tp:.2f}  R:R={rr:.2f}"
       if action != "HOLD" else "HOLD — no trade"),
    color=C["text"], fontsize=11,
)
_candles(ax, dec_df)

if action != "HOLD":
    _hline(ax, entry, C["entry"], f"ENTRY {entry:.2f}", lw=1.6, ls="-")
    if sl: _hline(ax, sl, C["sl"], f"SL  {sl:.2f}", lw=1.2)
    if tp: _hline(ax, tp, C["tp"], f"TP  {tp:.2f}", lw=1.2)
    if sl and tp:
        ax.axhspan(min(entry,sl), max(entry,sl), color=C["sl"], alpha=0.07, zorder=1)
        ax.axhspan(min(entry,tp), max(entry,tp), color=C["tp"], alpha=0.07, zorder=1)

acol = C["tp"] if action == "BUY" else C["sl"] if action == "SELL" else C["text"]
ax.text(0.99, 0.97, f"▶ {action}", transform=ax.transAxes, color=acol,
        fontsize=20, fontweight="bold", ha="right", va="top", alpha=0.75)

rsn = (dec.get("reasoning") or "").strip()
if rsn and rsn != "LLM not reachable":
    snippet = rsn[:160] + ("…" if len(rsn) > 160 else "")
    ax.text(0.01, 0.02, f'"{snippet}"', transform=ax.transAxes,
            color=C["text"], fontsize=7.5, va="bottom", alpha=0.7,
            bbox=dict(facecolor=C["bg"], edgecolor=C["grid"], alpha=0.85, pad=3))

ax.legend(loc="upper left", fontsize=8, framealpha=0.3,
          facecolor=C["bg"], edgecolor=C["grid"])
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════
print("STEP 6 · Simulate outcome")
print("─" * 56)
try:
    from pipeline import simulate_outcome, MAX_HOLD_DAYS, compute_pnl, trade_costs
    entry_p = cur_price   # 5M close — most recent price
    fwd = ta.load_forward(SYMBOL, DATE, MAX_HOLD_DAYS)
    exit_reason, exit_p, exit_date, _ = simulate_outcome(
        action, entry_p, sl, tp, fwd,
        max_hold=MAX_HOLD_DAYS, planned_entry=entry,
    )
    if exit_reason not in ("ENTRY_NOT_FILLED", "HOLD") and action != "HOLD" and sl and tp:
        gross = compute_pnl(action, entry, exit_p, 1)
        pnl   = round(gross - trade_costs(entry, exit_p, 1, action), 2)
    else:
        pnl = 0.0

    print(f"  Exit reason : {exit_reason}")
    print(f"  Exit price  : {exit_p:.2f}  (on {exit_date})")
    print(f"  P&L (qty=1) : {pnl:+.2f}")

    # ── Resample forward bars to 5M-day daily for charting ──────
    if len(fwd) > MAX_HOLD_DAYS * 2:
        fwd_d = (fwd.resample("1D")
                   .agg({"open":"first","high":"max","low":"min",
                         "close":"last","volume":"sum"})
                   .dropna(subset=["close"]))
    else:
        fwd_d = fwd.copy()

    # Combine: last 10 1H historical bars (daily resampled) + forward bars
    hist_1h_daily = (df_1h.resample("1D")
                         .agg({"open":"first","high":"max","low":"min",
                               "close":"last","volume":"sum"})
                         .dropna(subset=["close"]))
    hist_part = hist_1h_daily.tail(10).copy()
    fwd_d.index = fwd_d.index.normalize()
    hist_part.index = hist_part.index.normalize()
    fwd_only = fwd_d[~fwd_d.index.isin(hist_part.index)]
    combo = pd.concat([hist_part, fwd_only])
    n_hist = len(hist_part)

    fig, ax = plt.subplots(figsize=(14, 7))
    mark = "PROFIT" if pnl > 0 else "LOSS" if pnl < 0 else "FLAT"
    fig.suptitle(
        f"Step 6 · {SYMBOL} · {exit_reason} @ {exit_p:.2f}  [{mark}  {pnl:+.2f}]",
        color=C["text"], fontsize=11,
    )
    _candles(ax, combo)

    ax.axvline(n_hist - 0.5, color="#ffffff", lw=0.9, ls="--", alpha=0.4, zorder=5)
    ax.text(n_hist - 0.3, ax.get_ylim()[1], "  → entry day", color=C["text"],
            fontsize=7.5, va="top")

    if action != "HOLD":
        ax.axhline(entry, color=C["entry"], lw=1.5, ls="-",  alpha=0.9, zorder=4,
                   label=f"Entry {entry:.2f}")
        if sl: ax.axhline(sl, color=C["sl"], lw=1.2, ls="--", alpha=0.85, zorder=4,
                          label=f"SL {sl:.2f}")
        if tp: ax.axhline(tp, color=C["tp"], lw=1.2, ls="--", alpha=0.85, zorder=4,
                          label=f"TP {tp:.2f}")

    if exit_date:
        try:
            xi = _xi(combo, exit_date)
            ecol = C["tp"] if exit_reason == "TP_HIT" else C["sl"] if exit_reason == "SL_HIT" else C["entry"]
            ax.plot(xi, exit_p, "*", color=ecol, ms=16, zorder=7, label=f"{exit_reason}")
            ax.annotate(f"  {exit_reason}\n  {exit_p:.2f}",
                        xy=(xi, exit_p), xytext=(xi + 0.5, exit_p),
                        color=ecol, fontsize=8, va="center", zorder=7,
                        arrowprops=dict(arrowstyle="-", color=ecol, lw=0.7))
        except Exception:
            pass

    pcol = C["tp"] if pnl >= 0 else C["sl"]
    ax.text(0.99, 0.97, f"P&L  {pnl:+.2f}",
            transform=ax.transAxes, color=pcol,
            fontsize=16, fontweight="bold", ha="right", va="top", alpha=0.85)

    ax.legend(loc="upper left", fontsize=8, framealpha=0.3,
              facecolor=C["bg"], edgecolor=C["grid"])
    plt.tight_layout(); plt.show()

except Exception as _err:
    import traceback
    print(f"[outcome simulation failed: {_err}]")
    traceback.print_exc()
